In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import load_model

# ==============================
# Load Models
# ==============================
face_model = YOLO(r"D:\projects\facec detection\Face_detection\Face detection_enhanced\train\weights\best.pt")
emotion_model = load_model("best_emotion_model_scratch_enhanced.keras",
                           custom_objects={"preprocess_input": preprocess_input})

emotions = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

# ==============================
# Load Video
# ==============================
video_path = r"D:\projects\facec detection\Face_detection\image class emotion\Test.mp4"  # مسار الفيديو
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video.")
    exit()

# Get video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Output video writer
out_path = r"D:\projects\facec detection\Output_Test.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(out_path, fourcc, fps, (frame_width, frame_height))

# ==============================
# Process Video Frames
# ==============================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detect faces with YOLO
    results = face_model(frame)

    for box in results[0].boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)
        face = frame[y1:y2, x1:x2]
        if face.size == 0:
            continue

        # Preprocess for emotion model
        face = cv2.resize(face, (48, 48))
        #face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
        face = face / 255.0
        face = np.reshape(face, (1, 48, 48, 3))

        # Predict emotion
        prediction = emotion_model.predict(face, verbose=0)
        emotion_label = np.argmax(prediction)
        confidence = np.max(prediction)
        emotion_text = f"{emotions[emotion_label]} ({confidence:.2f})"

        # Draw bounding box and label
        y_text = y1 - 10 if y1 - 10 > 10 else y1 + 20
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, emotion_text, (x1, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # Write the frame to output video
    out.write(frame)

    # Optional: Show frame real-time
    cv2.imshow("Video Emotion Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release everything
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Output video saved to: {out_path}")


0: 640x384 1 Face, 38.9ms
Speed: 2.2ms preprocess, 38.9ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Face, 31.1ms
Speed: 1.6ms preprocess, 31.1ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Face, 29.9ms
Speed: 1.2ms preprocess, 29.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Face, 27.8ms
Speed: 1.2ms preprocess, 27.8ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Face, 30.7ms
Speed: 1.7ms preprocess, 30.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Face, 32.2ms
Speed: 1.1ms preprocess, 32.2ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Face, 34.7ms
Speed: 1.0ms preprocess, 34.7ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Face, 25.3ms
Speed: 1.1ms preprocess, 25.3ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x